In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

tickers = ["TSLA", "BND", "SPY"]
prices = pd.DataFrame({
    t: pd.read_csv(f"../data/processed/{t}.csv", index_col="Date", parse_dates=True)["Close"]
    for t in tickers
})
prices = prices.asfreq("B").ffill()

# Backtest period: last 12 months of available data
backtest = prices[prices.index >= (prices.index.max() - pd.Timedelta(days=365))]
returns = backtest.pct_change().dropna()

# Strategy weights from Task 4 (replace with your actual cleaned_weights values)
strategy_weights = {"TSLA": cleaned_weights["TSLA"], "BND": cleaned_weights["BND"], "SPY": cleaned_weights["SPY"]}
benchmark_weights = {"TSLA": 0.0, "BND": 0.40, "SPY": 0.60}

def portfolio_cumulative_return(returns_df, weights):
    weighted_returns = sum(returns_df[t] * w for t, w in weights.items())
    return (1 + weighted_returns).cumprod()

strategy_cum = portfolio_cumulative_return(returns, strategy_weights)
benchmark_cum = portfolio_cumulative_return(returns, benchmark_weights)

plt.figure(figsize=(12, 6))
plt.plot(strategy_cum.index, strategy_cum, label="Optimized Strategy")
plt.plot(benchmark_cum.index, benchmark_cum, label="Benchmark (60% SPY / 40% BND)")
plt.title("Backtest: Strategy vs Benchmark (Last 12 Months)")
plt.ylabel("Cumulative Growth (1.0 = starting value)")
plt.legend()
plt.savefig("../data/processed/backtest.png", dpi=150)
plt.show()

# Summary stats
strategy_total_return = strategy_cum.iloc[-1] - 1
benchmark_total_return = benchmark_cum.iloc[-1] - 1
strategy_sharpe = (returns.dot(pd.Series(strategy_weights)).mean() / returns.dot(pd.Series(strategy_weights)).std()) * np.sqrt(252)
benchmark_sharpe = (returns.dot(pd.Series(benchmark_weights)).mean() / returns.dot(pd.Series(benchmark_weights)).std()) * np.sqrt(252)

print(f"Strategy Total Return: {strategy_total_return:.2%}, Sharpe: {strategy_sharpe:.2f}")
print(f"Benchmark Total Return: {benchmark_total_return:.2%}, Sharpe: {benchmark_sharpe:.2f}")